# SST Device Selection Optimizer
## Chapter 4 - Solid-State Transformers

This notebook explores the fundamental trade-off in SST design: **which SiC device voltage class is optimal for a given MV grid voltage and power level?**

The key insight from the ETH Zurich dissertation is that the answer **depends critically on the operating point**:
- At 25 kW / 3.8 kV -> **10 kV SiC wins** (single-cell, highest efficiency)
- At 1 MW / 13.8 kV -> **3.3 kV SiC wins** (multi-cell ISOP, best efficiency + supply chain)
- At 10 MW / 34.5 kV -> **10 kV+ SiC returns** (cell-count reducer, post-2029)

### Physics Model

The specific on-resistance of SiC MOSFETs scales as:

$$R_{DS,on} \propto V_B^{2.5}$$

where $V_B$ is the blocking voltage. Higher-voltage devices have **exponentially worse conduction losses** per unit area, but reduce the number of series cells needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

DEVICES = {
    1.2: {'v_block': 1200,  'i_die': 202, 'rds_on': 0.045, 'cost': 8,   'name': '1200V SiC'},
    1.7: {'v_block': 1700,  'i_die': 150, 'rds_on': 0.072, 'cost': 15,  'name': '1700V SiC'},
    3.3: {'v_block': 3300,  'i_die': 52,  'rds_on': 0.160, 'cost': 35,  'name': '3300V SiC'},
    6.5: {'v_block': 6500,  'i_die': 35,  'rds_on': 0.420, 'cost': 120, 'name': '6500V SiC'},
    10.0:{'v_block': 10000, 'i_die': 20,  'rds_on': 0.850, 'cost': 450, 'name': '10kV SiC'},
}

def compute_sst_metrics(mv_kv, power_mw, dev_kv):
    d = DEVICES[dev_kv]
    v_ln_peak = mv_kv * 1000 / np.sqrt(3) * np.sqrt(2)
    v_dc = v_ln_peak * 1.35
    series_cells = max(1, int(np.ceil(v_dc / (d['v_block'] * 0.6))))
    cell_power = power_mw * 1e6 / (series_cells * 3)
    cell_current = cell_power / (d['v_block'] * 0.5)
    parallel_dies = max(1, int(np.ceil(cell_current / d['i_die'])))
    p_cond = cell_current**2 * d['rds_on'] / parallel_dies * series_cells * 3 / 1000
    f_sw = 50e3
    e_sw = 0.003 * (d['v_block'] / 1000) * power_mw
    p_sw = e_sw * 1e-3 * f_sw * series_cells * 3 / 1000
    total_loss = p_cond + p_sw
    efficiency = power_mw * 1000 / (power_mw * 1000 + total_loss) * 100
    total_dies = series_cells * parallel_dies * 3
    return {'series_cells': series_cells, 'parallel_dies': parallel_dies,
            'total_dies': total_dies, 'efficiency': efficiency,
            'p_cond': p_cond, 'p_sw': p_sw, 'v_dc': v_dc,
            'cost': total_dies * d['cost']}

## 1. Efficiency vs Device Class - The Core Trade-off

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ops = [(3.8, 0.025, '25kW/3.8kV (2019)'), (13.8, 1.0, '1MW/13.8kV (OCP v0.3)'), (34.5, 10.0, '10MW/34.5kV (Future)')]
dev_classes = [1.2, 1.7, 3.3, 6.5, 10.0]
colors = ['#0066FF', '#00FFCC', '#FFB800', '#FF6600', '#FF4444']

for ax, (mv, pwr, title) in zip(axes, ops):
    effs, cells_list = [], []
    for dv in dev_classes:
        m = compute_sst_metrics(mv, pwr, dv)
        effs.append(m['efficiency'])
        cells_list.append(m['series_cells'])
    bars = ax.bar(range(len(dev_classes)), effs, color=colors, alpha=0.8, edgecolor='white')
    ax.set_xticks(range(len(dev_classes)))
    ax.set_xticklabels([f'{d}kV' for d in dev_classes])
    ax.set_ylabel('Efficiency (%)')
    ax.set_title(title, fontsize=11)
    ax.set_ylim(min(effs)-1, max(effs)+0.5)
    ax.axhline(y=98, color='green', linestyle='--', alpha=0.5)
    for bar, eff, c in zip(bars, effs, cells_list):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                f'{eff:.2f}%\n({c} cells)', ha='center', va='bottom', fontsize=9)
    best_idx = np.argmax(effs)
    bars[best_idx].set_edgecolor('#00FF00')
    bars[best_idx].set_linewidth(3)
plt.suptitle('SST Efficiency by SiC Device Voltage Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../docs/assets/sst-device-scaling.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Efficiency Heatmap: MV Voltage x Power Level

In [ ]:
mv_range = np.linspace(3, 35, 50)
pwr_range = np.linspace(0.025, 10, 50)
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, dev_kv in zip(axes, [1.7, 3.3, 10.0]):
    eff_map = np.zeros((len(pwr_range), len(mv_range)))
    for i, pwr in enumerate(pwr_range):
        for j, mv in enumerate(mv_range):
            eff_map[i, j] = compute_sst_metrics(mv, pwr, dev_kv)['efficiency']
    im = ax.imshow(eff_map, aspect='auto', origin='lower',
                   extent=[mv_range[0], mv_range[-1], pwr_range[0], pwr_range[-1]],
                   cmap='RdYlGn', vmin=90, vmax=99.5)
    ax.set_xlabel('MV Grid Voltage (kV)')
    ax.set_ylabel('Power (MW)')
    ax.set_title(f'{dev_kv}kV SiC', fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, label='Efficiency (%)', shrink=0.8)
plt.suptitle('SST Efficiency Design Space', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../docs/assets/sst-efficiency-heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Optimal Device Selector Map

For any operating point, find the device class that maximizes efficiency.

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
mv_range = np.linspace(3, 35, 80)
pwr_range = np.linspace(0.025, 10, 80)
optimal_map = np.zeros((len(pwr_range), len(mv_range)))
eff_at_opt = np.zeros((len(pwr_range), len(mv_range)))
dev_list = [1.2, 1.7, 3.3, 6.5, 10.0]

for i, pwr in enumerate(pwr_range):
    for j, mv in enumerate(mv_range):
        best_eff, best_dev = 0, 0
        for dv in dev_list:
            m = compute_sst_metrics(mv, pwr, dv)
            if m['efficiency'] > best_eff:
                best_eff = m['efficiency']
                best_dev = dv
        optimal_map[i, j] = best_dev
        eff_at_opt[i, j] = best_eff

cmap = ListedColormap(['#0066FF', '#00FFCC', '#FFB800', '#FF6600', '#FF4444'])
bounds = [0.5, 1.5, 2.5, 4.0, 8.0, 12.0]
norm = BoundaryNorm(bounds, cmap.N)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
im1 = ax1.imshow(optimal_map, aspect='auto', origin='lower',
                 extent=[mv_range[0], mv_range[-1], pwr_range[0], pwr_range[-1]],
                 cmap=cmap, norm=norm, interpolation='nearest')
ax1.set_xlabel('MV Grid Voltage (kV)')
ax1.set_ylabel('Power (MW)')
ax1.set_title('Optimal Device Class', fontweight='bold')
cbar1 = plt.colorbar(im1, ax=ax1, label='Device Class (kV)', shrink=0.8)
cbar1.set_ticks([1.2, 1.7, 3.3, 6.5, 10.0])
for mv, pwr, label in [(3.8, 0.025, 'ETH 2019'), (13.8, 1.0, 'OCP v0.3'), (34.5, 5.0, 'Future')]:
    ax1.plot(mv, pwr, 'w*', markersize=15, markeredgecolor='black')
    ax1.annotate(label, (mv, pwr), textcoords='offset points', xytext=(10, 5), fontsize=8, color='white', fontweight='bold')

im2 = ax2.imshow(eff_at_opt, aspect='auto', origin='lower',
                 extent=[mv_range[0], mv_range[-1], pwr_range[0], pwr_range[-1]],
                 cmap='RdYlGn', vmin=95, vmax=99.5)
ax2.set_xlabel('MV Grid Voltage (kV)')
ax2.set_ylabel('Power (MW)')
ax2.set_title('Efficiency at Optimal Device', fontweight='bold')
plt.colorbar(im2, ax=ax2, label='Efficiency (%)', shrink=0.8)
plt.suptitle('SST Device Selection Optimizer', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../docs/assets/sst-device-optimizer.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key findings:')
for mv, pwr, label in [(3.8, 0.025, '25kW/3.8kV'), (13.8, 1.0, '1MW/13.8kV'), (34.5, 10.0, '10MW/34.5kV')]:
    best = max(dev_list, key=lambda dv: compute_sst_metrics(mv, pwr, dv)['efficiency'])
    m = compute_sst_metrics(mv, pwr, best)
    print(f'  {label}: optimal={best}kV ({m["efficiency"]:.2f}%, {m["series_cells"]} cells/phase)')